# PerfectPetClub — Task 2: ETL Pipeline for Customer Churn
**Module:** 7COM2005 Data Engineering | **Team:** P4G5 | **Notebook:** `notebooks/etl_pipeline.ipynb`

**Goal:** build a clean, one-row-per-customer dataset for analysing **churn** (a customer cancelling the auto-renew of their subscription) and write it to Parquet for the PySpark ML notebook (Task 3).

### Pipeline architecture
```
EXTRACT                         TRANSFORM                                   LOAD
SQLite (9 tables)  ─┐           clean_* functions  (types, DQ checks)
CRM report (CSV)   ─┤──►  raw   ─► build_features()  (feature engineering) ─► validate ─► data/processed/
Server log (CSV)   ─┤           ─► leakage audit + final validation                         processed_churn.parquet
Survey (CSV)       ─┘
```

### Key design decisions (be ready to justify these)
1. **Target:** `churn = 1 - auto_renew` (the brief defines churn as cancelling auto-renew). 
2. **Data-leakage prevention:** `subscription.end_date`, tenure, "days since last order" and raw activity *counts* are **deliberately excluded**. Customers who cancel stop ordering and logging in, so these columns would reveal the label (a model would look brilliant but be useless). All behavioural features are **rates per month of each customer's own observation window**, or ratios/trends, so they do not encode how long someone stayed.
3. **No PII in the output:** name, e-mail, password hash and address are dropped before loading.
4. **Missing delivery dates are flagged, not invented:** 1,090 orders (4.7%) have no expected/delivered date, spread across the whole year (so not only recent orders). We cannot tell whether they are in transit or simply unrecorded, so we create a `delivery_missing` flag instead of fabricating dates.